In [0]:
%sql
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # Gold layer - Build dimensions
-- MAGIC
-- MAGIC This notebook creates the dimension tables from the cleaned Silver OBT.
-- MAGIC
-- MAGIC I use `sha2()` hash IDs instead of `dense_rank()` because the lecturer said
-- MAGIC `dense_rank()` may not work well with streaming-style tables.
-- MAGIC
-- MAGIC Source table:
-- MAGIC - marathos.silver.races_obt
-- MAGIC
-- MAGIC Target tables:
-- MAGIC - marathos.gold.dim_event
-- MAGIC - marathos.gold.dim_athlete
-- MAGIC - marathos.gold.dim_date

-- COMMAND ----------

CREATE OR REPLACE TABLE marathos.gold.dim_event AS
SELECT DISTINCT
  event_id,
  event_name,
  event_year,
  event_dates,
  event_distance_raw,
  distance_km,
  distance_unit,
  event_finishers
FROM marathos.silver.races_obt
WHERE event_id IS NOT NULL
  AND event_name IS NOT NULL
  AND event_year IS NOT NULL;

-- COMMAND ----------

CREATE OR REPLACE TABLE marathos.gold.dim_athlete AS
SELECT DISTINCT
  athlete_id,
  athlete_source_id,
  athlete_country,
  athlete_gender,
  athlete_birth_year,
  athlete_age,
  athlete_age_category,
  athlete_club
FROM marathos.silver.races_obt
WHERE athlete_id IS NOT NULL;

-- COMMAND ----------

CREATE OR REPLACE TABLE marathos.gold.dim_date AS
SELECT DISTINCT
  sha2(concat_ws('|',
    cast(event_year as string),
    coalesce(event_dates, 'unknown')
  ), 256) AS date_id,
  event_year,
  event_dates
FROM marathos.silver.races_obt
WHERE event_year IS NOT NULL;

-- COMMAND ----------

SELECT 'dim_event' AS table_name, COUNT(*) AS row_count
FROM marathos.gold.dim_event

UNION ALL

SELECT 'dim_athlete' AS table_name, COUNT(*) AS row_count
FROM marathos.gold.dim_athlete

UNION ALL

SELECT 'dim_date' AS table_name, COUNT(*) AS row_count
FROM marathos.gold.dim_date;